In [1]:
! pip install duckdb

Defaulting to user installation because normal site-packages is not writeable
  Using cached duckdb-1.5.1-cp310-cp310-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl (21.3 MB)


In [2]:
import duckdb
import logging 
import time
import pandas as pd
import os

In [55]:
logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    filename='pipeline.log'
)
logger = logging.getLogger(__name__)

In [56]:
#data preparation 
def load_incidents(con):
    try:
        con.execute("""
            CREATE TABLE IF NOT EXISTS incidents (
                cad_incident_id VARCHAR PRIMARY KEY,
                incident_datetime TIMESTAMP,
                incident_close_datetime TIMESTAMP,
                final_call_type VARCHAR,
                zipcode VARCHAR,
                initial_call_type VARCHAR,
                borough VARCHAR
            )
        """)
        logger.info("Created incidents table")

        con.execute("""
            INSERT INTO incidents
                SELECT
                    cad_incident_id,
                    incident_datetime::TIMESTAMP,
                    incident_close_datetime::TIMESTAMP,
                    final_call_type,
                    zipcode,
                    initial_call_type,
                    borough
                FROM read_parquet('data/EMS_Incidents.parquet')
        """)

        row_count = con.execute("SELECT COUNT(*) FROM incidents").fetchone()[0]
        logger.info(f"Loaded incidents: {row_count:,} rows")
        print(f"Loaded incidents: {row_count:,} rows")

    except Exception as e:
        logger.error(f"Failed to load incidents: {e}")
        print(f"Failed to load incidents: {e}")

In [57]:
def load_events(con):
    try:
        con.execute("""
            CREATE TABLE IF NOT EXISTS events (
                event_id VARCHAR PRIMARY KEY,
                start_date_time TIMESTAMP,
                end_date_time TIMESTAMP,
                event_type VARCHAR,
                event_name VARCHAR,
                event_borough VARCHAR
            )
        """)
        logger.info("Created events table")

        con.execute("""
            INSERT INTO events
                SELECT
                    CAST(event_id AS VARCHAR),
                    start_date_time::TIMESTAMP,
                    end_date_time::TIMESTAMP,
                    event_type,
                    event_name,
                    event_borough
                FROM (
                    SELECT *,
                        ROW_NUMBER() OVER (
                            PARTITION BY event_id
                            ORDER BY start_date_time
                        ) AS rn
                    FROM read_parquet('data/NYC_Events.parquet'))
                WHERE rn = 1
        """)

        row_count = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
        logger.info(f"Loaded events: {row_count:,} rows")
        print(f"Loaded events: {row_count:,} rows")

    except Exception as e:
        logger.error(f"Failed to load events: {e}")
        print(f"Failed to load events: {e}")

In [58]:
def load_weather(con):
    try:
        #use sequence to create weather id 
        con.execute("""
            CREATE TABLE IF NOT EXISTS weather (
                datetime TIMESTAMP,
                temperature DOUBLE,
                weathercode DOUBLE,
                borough VARCHAR
            )
        """)
        logger.info("Created weather table")

        con.execute("""
            INSERT INTO weather
                SELECT
                    datetime::TIMESTAMP,
                    temperature,
                    weathercode,
                    borough
                FROM read_parquet('data/Weather.parquet')
        """)

        row_count = con.execute("SELECT COUNT(*) FROM weather").fetchone()[0]
        logger.info(f"Loaded weather: {row_count:,} rows")
        print(f"Loaded weather: {row_count:,} rows")

    except Exception as e:
        logger.error(f"Failed to load weather: {e}")
        print(f"Failed to load weather: {e}")



In [59]:
def load_demographics(con):
    try:
        con.execute("""
            CREATE TABLE IF NOT EXISTS demographics (
                region_name VARCHAR PRIMARY KEY,
                year INTEGER,
                crime_viol_rt   DOUBLE,
                hh_inc_med_adj  DOUBLE,
                hh_u18_pct      DOUBLE,
                pop_65p_pct     DOUBLE,
                pop_num         DOUBLE
            )
        """)
        logger.info("Created demographics table")

        con.execute("""
            INSERT INTO demographics
                SELECT
                    region_name, 
                    year, 
                    crime_viol_rt::DOUBLE,
                    hh_inc_med_adj::DOUBLE,
                    hh_u18_pct::DOUBLE,
                    pop_65p_pct::DOUBLE,
                    CAST(REPLACE(pop_num, ',', '') AS DOUBLE)
                FROM (SELECT *, ROW_NUMBER() OVER (
                        PARTITION BY region_name
                        ORDER BY year DESC
                    ) AS rn
                FROM read_parquet('data/Demographics.parquet'))
                WHERE rn = 1
        """)

        row_count = con.execute("SELECT COUNT(*) FROM demographics").fetchone()[0]
        logger.info(f"Loaded demographics: {row_count:,} rows")
        print(f"Loaded demographics: {row_count:,} rows")

    except Exception as e:
        logger.error(f"Failed to load demographics: {e}")
        print(f"Failed to load demographics: {e}")

In [62]:
if __name__ == "__main__":
    con = duckdb.connect(database='ems.duckdb', read_only=False)
    load_incidents(con)
    load_events(con)
    load_weather(con)
    load_demographics(con)
    con.close()

ConnectionException: Connection Error: Can't open a connection to same database file with a different configuration than existing connections

In [ ]:
#query
con = duckdb.connect(database='ems.duckdb', read_only=True)
result = con.execute("""
    WITH incident_counts AS (
    SELECT
        borough,
        DATE_TRUNC('hour', incident_datetime) AS dt,
        COUNT(*) AS incident_count
    FROM incidents
    GROUP BY borough, dt
    ),

    weather_features AS (
        SELECT
            borough,
            DATE_TRUNC('hour', datetime) AS dt,
            AVG(temperature) AS temperature
        FROM weather
        GROUP BY borough, dt
    ),

    event_features AS (
        SELECT
            event_borough,
            DATE_TRUNC('hour', start_date_time) AS dt,
            COUNT(*) AS num_events
        FROM events
        GROUP BY event_borough, dt
    )

    SELECT
        i.borough,
        i.dt,
        i.incident_count,

        -- time features
        EXTRACT(hour FROM i.dt) AS hour,
        EXTRACT(dow FROM i.dt) AS day_of_week,
        EXTRACT(month FROM i.dt) AS month,

        -- weather
        w.temperature,

        -- events
        COALESCE(e.num_events, 0) AS num_events,

        -- demographics
        d.pop_num,
        d.hh_inc_med_adj

    FROM incident_counts i

    LEFT JOIN weather_features w
    ON LOWER(TRIM(i.borough)) = LOWER(TRIM(w.borough))

    LEFT JOIN event_features e
    ON LOWER(TRIM(i.borough)) = LOWER(TRIM(e.event_borough)) AND i.dt = e.dt

    LEFT JOIN demographics d
    ON LOWER(TRIM(i.borough)) = LOWER(TRIM(d.region_name))
""").fetchdf()

In [54]:
result.head()

,borough,dt,incident_count,hour,day_of_week,month,temperature,num_events,pop_num,hh_inc_med_adj
0,QUEENS,NaT,1152673,<NA>,<NA>,<NA>,NaN,0,NaN,NaN
1,MANHATTAN,NaT,1515789,<NA>,<NA>,<NA>,NaN,0,NaN,NaN
2,RICHMOND / STATEN ISLAND,NaT,247945,<NA>,<NA>,<NA>,NaN,0,NaN,NaN
3,UNKNOWN,NaT,37,<NA>,<NA>,<NA>,NaN,0,NaN,NaN
4,BROOKLYN,NaT,1673397,<NA>,<NA>,<NA>,NaN,0,NaN,NaN


In [ ]:
#solution analysis
X = df.drop(columns=["incident_count", "dt"])
y = df["incident_count"]

# one-hot encode borough
X = pd.get_dummies(X, columns=["borough"], drop_first=True)

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100)
model.fit(X, y)

sample = {
    "borough": "Manhattan",
    "hour": 14,
    "day_of_week": 2,
    "month": 6,
    "temperature": 75,
    "num_events": 3,
    "pop_num": 1600000,
    "hh_inc_med_adj": 85000
}

analysis rationale